# Settings

In [1]:
parameters_file_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/params.json"
convert_complex = False

#Simulation settings
using_simulation_data = False
rf_matrix_path = "/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/rf0001.mat"

#Real data settings
using_matlab_data = True       #! high RAM usage for big files -> convert mat file to npy for memory map usage and set using_simulation_data to False
interpolate_data = True

#Batch processing settings
batch_processing = True
batch_size = 160                 #ignored if batch_processing is False
frames_to_process = 160
MAX_WORKERS = 10                 #max is 12 for the current computer (10 max recommended)

#Beamforming settings
run_nsi_apodization = False
nsi_dc_offset = 0.01
other_apodization = "rectangle".lower()

#Saving parameters
save_image = True
bf_data_path = "./data_bf/"
folder_name = "results_gab"
beamformed_apodized_matrix_name = "003_bf"
angles_debbuging = False          #if true saves the angles used for the beamforming (z,x, angles, frames)


# Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import json
import timeit
import os
import math
import concurrent.futures
from numba import njit
from scipy.signal import envelope, firwin, hilbert
from scipy import fft
from scipy.io import loadmat

import modules.beamforming as bf

# Import parameters

In [3]:
def load_parameters(json_file_path):
    with open(json_file_path, 'r') as f:
        params = json.load(f)
    return params


In [4]:
def add_extra_params(params):
    
    params["rf"]["upsampled_samples"] = params["rf"]["samples"] + (params["rf"]["interpolation_factor"] - 1) * (params["rf"]["samples"] - 1)
    params["rf"]["upsampled_sampling_rate"] = params["rf"]["sampling_rate"] * params["rf"]["interpolation_factor"] 

    return params

In [5]:
params = load_parameters(parameters_file_path)
params = add_extra_params(params)

pos = np.linspace(start=0, stop=params["array"]["length"], num=params["array"]["length"], dtype=int) #array position sampling
time = np.linspace(start=0, stop=1 / params["rf"]["sampling_rate"] * params["rf"]["samples"], num=params["rf"]["samples"]) #time sampling
C = params["rf"]["C"] #speed of sound

print(f"Max acquisition depth : {params["array"]["acq_depth"][1]} m") #v = d / t => d = v * t / 2 (round trip) with t = sample_step * samples

Max acquisition depth : 0.012 m


# Create workspace

In [6]:
if not os.path.exists(f"{bf_data_path}{folder_name}"):
    os.makedirs(f"{bf_data_path}{folder_name}")
    print(f"Folder {bf_data_path}{folder_name} doesn't exist. Setting up the folder.")

os.makedirs(f"{bf_data_path}{folder_name}/beamformed_final", exist_ok=True)
os.makedirs(f"{bf_data_path}{folder_name}/beamformed_apodized", exist_ok=True)
os.makedirs(f"{bf_data_path}{folder_name}/images", exist_ok=True)

# Run beamforming

In [7]:
beamforming = bf.Beamforming(
    nsi_dc_offset=nsi_dc_offset,
    params=params, 

    debug=False,        #! if want debug, set cuda flag to False
    angles_debug=False,

    MAX_WORKERS=MAX_WORKERS, 
    frames_to_process=frames_to_process, 
    batch_size=batch_size, 

    rf_matrix_path=rf_matrix_path, 
    using_simulation_data=using_simulation_data, 
    using_matlab_data=using_matlab_data, 
    convert_complex=convert_complex,
    interpolate_data=interpolate_data,
    run_nsi_apodization=run_nsi_apodization,
    other_apodization=other_apodization,
    
    bf_data_path=bf_data_path,
    folder_name=folder_name,
    beamformed_apodized_matrix_name=beamformed_apodized_matrix_name,
    )

beamforming.run_beamforming(cuda=True)

Samples for acquisition depth: 200
Samples for array length: 256
Beamformed data of shape (200, 256, 160) saved at: ./data_bf/results_gab/beamformed_apodized/003_bf_rectangle.npy

Starting process pool with 1 workers.
CUDA enabled, processing on GPU.
Total frames to process: 160
Batch size: 160
Total batches: 1
Last batch identified: Start Frame 0, Size 160



Waiting for all batches to complete...

Processing batch 1 [1/1] (Process ID: 98853)
  • Processing from frame 0 to frame 160
  • Processing 160 frame(s)
  • RF matrix: Imported RF matrix with shape: (768, 128, 16, 160)
  • Running rectangle apodization

!!! Error in process 98853 processing batch 1 (frames 0 to 160): Failed in cuda mode pipeline (step: native lowering)
Failed in nopython mode pipeline (step: nopython frontend)
- Resolution failure for literal arguments:
No implementation of function Function(<function _ol_array_allocate at 0x785710b15800>) found for signature:

 >>> _ol_array_allocate(typeref[<class 'numba.core.types.npytypes.Array'>], int64, uint32)

There are 2 candidate implementations:
  - Of which 2 did not match due to:
  Overload in function '_ol_array_allocate': File: numba/np/arrayobj.py: Line 4286.
    With argument(s): '(typeref[<class 'numba.core.types.npytypes.Array'>], int64, uint32)':
   Rejected as the implementation raised a specific error:
     Numb

Traceback (most recent call last):
  File "/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/modules/beamforming.py", line 231, in process_batch_multiprocessing
    bf_apodized_matrix = self.beamforming_gpu(
        rf_matrix, actual_loaded_frames, apo_func
    )
  File "/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/modules/beamformer_gpu.py", line 334, in beamforming_gpu
    self._beamforming_gpu[(blocks_x, blocks_y, blocks_z), threads_per_block](
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        rf_device,
        ^^^^^^^^^^
    ...<19 lines>...
        t0_delay_device,
        ^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/matmat/miniconda3/envs/superresolution/lib/python3.13/site-packages/numba/cuda/dispatcher.py", line 539, in __call__
    return self.dispatcher.call(args, self.griddim, self.blockdim,
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                                self.stre

Batch 1 failed.

--- Processing Summary ---

Total Batches: 1
Successful Batches: 0
Failed Batches: 1

1 batch(es) failed. Check logs above.

Total Beamforming took: 7.10 seconds


# Apply NSI 

In [8]:
# def get_envelope(bf_matrix, frames):
#     lateral_resolution = bf_matrix.shape[1]
#     samples = bf_matrix.shape[0]

#     env = np.zeros((samples, lateral_resolution, frames), dtype=bf_matrix.dtype)
#     for i in range(0, frames):
#         for j in range(0, lateral_resolution):
#             env[:,j,i] = envelope(bf_matrix[:,j,i], residual=None)
#     return env

In [9]:
# if run_nsi_apodization:

#     bf_env_nsi = np.zeros((samples_for_acquisition_depth, samples_for_array_length, frames_to_process), dtype=bf_dtype)

#     memmap_bf_matrix_zm = np.load(f"{bf_data_path}{folder_name}/beamformed_apodized/{beamformed_apodized_matrix_name}_zm.npy")
#     memmap_bf_matrix_dc1 = np.load(f"{bf_data_path}{folder_name}/beamformed_apodized/{beamformed_apodized_matrix_name}_dc1.npy")
#     memmap_bf_matrix_dc2 = np.load(f"{bf_data_path}{folder_name}/beamformed_apodized/{beamformed_apodized_matrix_name}_dc2.npy")

#     memmap_bf_matrix_zm = get_envelope(memmap_bf_matrix_zm, frames=frames_to_process)
#     memmap_bf_matrix_dc1 = get_envelope(memmap_bf_matrix_dc1, frames=frames_to_process)
#     memmap_bf_matrix_dc2 = get_envelope(memmap_bf_matrix_dc2, frames=frames_to_process)

#     for i in range(0, frames_to_process):
#         bf_env_nsi[:,:,i] = (memmap_bf_matrix_dc1[:,:,i] + memmap_bf_matrix_dc2[:,:,i]) / 2 - memmap_bf_matrix_zm[:,:,i]

#     save_beamformed_data(bf_env_nsi, f"{beamformed_apodized_matrix_name}_env_nsi", destination=f"{bf_data_path}{folder_name}/beamformed_apodized/")
    
#     del memmap_bf_matrix_zm
#     del memmap_bf_matrix_dc1
#     del memmap_bf_matrix_dc2
#     del bf_env_nsi

# Sum frames

In [10]:
# bf_frames_path = os.path.join(bf_data_path, folder_name, "beamformed_apodized") 
# bf_frames = np.load(f"{bf_frames_path}/{beamformed_apodized_matrix_name}_{"env_nsi" if run_nsi_apodization else other_apodization}.npy")

# bf_final_path = os.path.join(bf_data_path, folder_name, "beamformed_final")
# bf_final = np.sum(bf_frames, axis=2)
# save_beamformed_data(bf_final, f"{beamformed_apodized_matrix_name}_{"env_nsi" if run_nsi_apodization else other_apodization}", destination=f"{bf_final_path}/")

# del bf_final
# del bf_frames

## Show beamformed image

In [11]:
# bf_image = np.load(f"{bf_data_path}{folder_name}/beamformed_final/{beamformed_apodized_matrix_name}_{"env_nsi" if run_nsi_apodization else other_apodization}.npy")

# pixel_width = params["image"]["dx"]
# pixel_height = params["image"]["dz"]
# physical_aspect = pixel_width / pixel_height

# plt.figure(figsize=(10, 8))
# plt.imshow(np.abs(bf_image), cmap='viridis', aspect=physical_aspect)

# plt.title(f"Beamformed Image ({'env_nsi' if run_nsi_apodization else other_apodization})")

# plt.xlabel("Lateral samples")
# plt.ylabel("Depth (samples)")

# plt.colorbar(label="Amplitude") 

# if save_image:
#     plt.savefig(f"{bf_data_path}{folder_name}/images/{beamformed_apodized_matrix_name}_{"env_nsi" if run_nsi_apodization else other_apodization}.png")
# plt.show()

In [12]:
# import numpy as np
# bf_up = np.load('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_up_rectangle.npy')
# bf_down = np.load('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_down_rectangle.npy')

# bf_both = bf_up + bf_down
# np.save('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_both_rectangle.npy', bf_both)

In [13]:
# import numpy as np
# bf_up_zm = np.load('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_up_zm.npy')
# bf_down_zm = np.load('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_down_zm.npy')
# bf_both_zm = bf_up_zm + bf_down_zm
# np.save('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_both_zm.npy', bf_both_zm)
# bf_up_dc1 = np.load('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_up_dc1.npy')
# bf_down_dc1 = np.load('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_down_dc1.npy')
# bf_both_dc1 = bf_up_dc1 + bf_down_dc1
# np.save('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_both_dc1.npy', bf_both_dc1)
# bf_up_dc2 = np.load('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_up_dc2.npy')
# bf_down_dc2 = np.load('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_down_dc2.npy')
# bf_both_dc2 = bf_up_dc2 + bf_down_dc2
# np.save('/home/matmat/Documents/urban_lab/002_Superresolution/superresolution/data_bf/mouse_prebf_filtered/beamformed_apodized/001_bf_IQ_testing_both_dc2.npy', bf_both_dc2)